# Telco Customer Churn: Data Understanding

This notebook loads the raw dataset, checks its structure and data quality, and explores the customer churn target.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'raw'
    / 'WA_Fn-UseC_-Telco-Customer-Churn.csv'
)

df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
# Initial data-quality check
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

print('\nMissing values:')
print(df.isnull().sum())

print('\nDuplicate rows:')
print(df.duplicated().sum())

print('\nTarget distribution:')
print(df['Churn'].value_counts())

print('\nTarget percentage:')
print(df['Churn'].value_counts(normalize=True) * 100)

## TotalCharges

In Phase 2 we'll investigate the values and convert this field to a numeric type if needed.

In [ ]:
df['TotalCharges'].dtype

In [ ]:
df['TotalCharges'].unique()[:20]

## Column groups

- **Identifier:** `customerID`
- **Categorical features:** customer demographics, services, contract, and billing choices
- **Numeric features:** `tenure`, `MonthlyCharges`, `TotalCharges`
- **Target:** `Churn`

In [ ]:
ID_COLUMN = 'customerID'

CATEGORICAL_COLUMNS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
]
NUMERIC_COLUMNS = ['tenure', 'MonthlyCharges', 'TotalCharges']
TARGET_COLUMN = 'Churn'

expected_columns = [ID_COLUMN, *CATEGORICAL_COLUMNS, *NUMERIC_COLUMNS, TARGET_COLUMN]
missing_columns = set(expected_columns) - set(df.columns)
assert not missing_columns, f'Missing columns: {missing_columns}'

pd.DataFrame({
    'column': expected_columns,
    'role': (['identifier'] + ['categorical'] * len(CATEGORICAL_COLUMNS)
             + ['numeric'] * len(NUMERIC_COLUMNS) + ['target']),
    'dtype': [df[column].dtype for column in expected_columns],
})

## Data quality

`TotalCharges` is commonly loaded as text because blank values occur in the source file. Convert it safely before analysing numeric values.

In [ ]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

quality_summary = pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_values': df.isna().sum(),
    'missing_percent': (df.isna().mean() * 100).round(2),
    'unique_values': df.nunique(),
})
quality_summary

In [ ]:
print('Duplicate customer IDs:', df[ID_COLUMN].duplicated().sum())
print('Duplicate rows:', df.duplicated().sum())

df[NUMERIC_COLUMNS].describe().T

## Churn distribution

Check the class balance before modelling, then compare churn rates across categorical features.

In [ ]:
churn_summary = (
    df[TARGET_COLUMN].value_counts(dropna=False)
    .rename_axis(TARGET_COLUMN)
    .reset_index(name='customers')
)
churn_summary['percent'] = (churn_summary['customers'] / len(df) * 100).round(2)
churn_summary

In [ ]:
churn_rate_by_feature = {}
for column in CATEGORICAL_COLUMNS:
    churn_rate_by_feature[column] = (
        pd.crosstab(df[column], df[TARGET_COLUMN], normalize='index')
        .mul(100)
        .round(1)
    )

# Example: compare churn rate by contract type.
churn_rate_by_feature['Contract']